# 转生千金 · 日语配音精准重抽与全量合包（1-Cell 智能增量重抽 + 同卡 anime-whisper 质检防崩版）

1. 在 Colab 左侧 **🔑 Secrets** 配置 `FISH_API_KEY` 并开启 Notebook 访问开关（零手动填表）。
2. 点击下方唯一代码单元格左上角 **▶ 运行**，一镜到底自动完成：自动检测 Google Drive 底包（已有 ≥800 条合格好句时保留 864 条原装好句，仅精准重抽 `tools/voice/retake_bad_ids.txt` 共 412 条；空目录时自动回退全量 1,276 条）→ 本地 sglang-omni FP8/INT4 或 Fish Audio API 推理 → 保守 `-55dB` 静音修剪与 `-16 LUFS` 响度统一 → 同卡 `litagin/anime-whisper` 实时 ASR 质检与坏句自动重抽 → 合并构建全量 1,276 条 Manifest 并打包下载 `ojousama_jp_voice_pack.zip`。

In [ ]:
# @title 🎙️ 转生千金 · 日语配音精准重抽与全量合包（1-Cell 智能增量重抽 + 同卡 anime-whisper 质检防崩版）
# 说明：直接从 Colab 左侧 🔑 Secrets 读取 FISH_API_KEY（零手动填表），自动检测 Google Drive 上一轮底包：
#       - 若已有 >= 800 条合格好句：保留全部 864 条原装好句绝不重跑，仅针对 tools/voice/retake_bad_ids.txt 精准重抽（~8-12 分钟）！
#       - 若为全新空目录：自动回退跑全量 1,276 条 + 同卡 anime-whisper 自动质检！
#       最终将全部 1,276 个 MP3 合并构建 Manifest、打包 /content/ojousama_jp_voice_pack.zip、备份云盘并触发浏览器下载。

from __future__ import annotations

import collections
import getpass
import importlib.util
import json
import os
import shutil
import signal
import subprocess
import sys
import time
import urllib.request
import zipfile
from pathlib import Path
from typing import IO

try:
    import tomllib
except ModuleNotFoundError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "tomli"])
    import tomli as tomllib  # type: ignore[no-redef]

# =========================
# 0. 全局常量与严格路径隔离
# =========================
REPO_URL = "https://github.com/juanhanyu/fictional-adventure.git"
REPO_DIR = Path("/content/fictional-adventure")

# 【防崩红线 #1】HF 缓存、模型权重、参考音 WAV 强制走 Colab 本地高速 NVMe SSD，严禁落入 /content/drive！
HF_HOME_DIR = Path("/content/hf_cache")
LOCAL_MODELS_ROOT = Path("/content/models")
LOCAL_REFS_DIR = Path("/content/refs_wav")
REFS_JSON = Path("/content/refs_resolved.json")
EFFECTIVE_RETAKE_IDS_PATH = Path("/content/retake_effective_ids.txt")
RETRY_FAILED_IDS_PATH = Path("/content/retry_failed_ids.txt")

SGL_SRC_DIR = Path("/content/sglang-omni")
SGL_VENV_DIR = Path("/content/sgl_venv")
SGL_OVERRIDES_PATH = Path("/content/omni-overrides.txt")
SERVE_LOG_PATH = Path("/content/sgl_omni_serve.log")
LOCAL_ZIP_PATH = Path("/content/ojousama_jp_voice_pack.zip")

SERVE_HOST = "127.0.0.1"
SERVE_PORT = 8000
SGLANG_SPEECH_URL = f"http://{SERVE_HOST}:{SERVE_PORT}/v1/audio/speech"
SGLANG_HEALTH_URL = f"http://{SERVE_HOST}:{SERVE_PORT}/health"
STARTUP_TIMEOUT_SEC = 1200  # 首次编译 Triton / FlashInfer / CUDA Graph 留足 20 分钟余量

ANIME_WHISPER_REPO_ID = "litagin/anime-whisper"
GOOD_BASE_PACK_MIN_COUNT = 800
FISH_API_FALLBACK_MODEL = "drama-3"

os.environ["PYTHONUNBUFFERED"] = "1"
os.environ["HF_HOME"] = str(HF_HOME_DIR)
os.environ["HUGGINGFACE_HUB_CACHE"] = str(HF_HOME_DIR / "hub")
os.environ.pop("TRANSFORMERS_CACHE", None)

for _ssd_path in (HF_HOME_DIR, LOCAL_MODELS_ROOT, LOCAL_REFS_DIR, REFS_JSON):
    assert not str(_ssd_path.resolve()).startswith("/content/drive"), (
        f"致命配置错误：高速本地路径 {_ssd_path} 绝不允许指向 Google Drive FUSE！"
    )

HF_HOME_DIR.mkdir(parents=True, exist_ok=True)
LOCAL_MODELS_ROOT.mkdir(parents=True, exist_ok=True)
LOCAL_REFS_DIR.mkdir(parents=True, exist_ok=True)


# =========================
# 1. 工具函数（流式日志 / 脱敏 / 精准进程与显存清理）
# =========================
def mask_cmd(cmd: list[str]) -> str:
    """打印命令行时自动脱敏 API Key，防止密钥泄露到笔记本输出。"""
    masked: list[str] = []
    hide_next = False
    for tok in cmd:
        if hide_next:
            masked.append("***")
            hide_next = False
        elif tok in ("--api-key", "--fish-api-key"):
            masked.append(tok)
            hide_next = True
        elif tok.startswith(("--api-key=", "--fish-api-key=")):
            k, _ = tok.split("=", 1)
            masked.append(f"{k}=***")
        else:
            masked.append(tok)
    return " ".join(masked)


def run_stream(
    cmd: list[str],
    *,
    cwd: Path | None = None,
    env: dict[str, str] | None = None,
    check: bool = True,
) -> int:
    """实时流式打印子进程输出（合并 stdout+stderr），异常退出时保留末尾 80 行完整上下文。"""
    print(f"\n$ {mask_cmd(cmd)}", flush=True)
    run_env = (env or os.environ).copy()
    run_env["PYTHONUNBUFFERED"] = "1"
    tail = collections.deque(maxlen=80)

    proc = subprocess.Popen(
        cmd,
        cwd=str(cwd) if cwd is not None else None,
        env=run_env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        encoding="utf-8",
        errors="replace",
        bufsize=1,
    )
    assert proc.stdout is not None
    for line in proc.stdout:
        print(line, end="", flush=True)
        tail.append(line.rstrip("\r\n"))
    rc = proc.wait()
    if check and rc != 0:
        tail_txt = "\n".join(tail) if tail else "(无输出)"
        raise RuntimeError(
            f"子进程执行失败 (exit={rc}): {mask_cmd(cmd)}\n"
            f"--- 子进程最后 {len(tail)} 行输出 ---\n{tail_txt}"
        )
    return rc


def read_log_tail(log_path: Path, n_lines: int = 150) -> str:
    """读取日志文件最后 n_lines 行（折叠 tqdm 回车刷屏，绝不截断单行长度，确保底部完整 Exception 可见）。"""
    if not log_path.is_file():
        return "(日志文件不存在)"
    raw = log_path.read_text(encoding="utf-8", errors="replace").replace("\r\n", "\n")
    lines = [ln.rsplit("\r", 1)[-1] for ln in raw.split("\n")]
    while lines and not lines[-1].strip():
        lines.pop()
    if not lines:
        return "(日志文件为空)"
    return "\n".join(lines[-n_lines:])


def _find_listening_inodes_for_port(port: int) -> set[str]:
    """从 /proc/net/tcp{,6} 查找处于 LISTEN (0A) 状态的指定端口 socket inode。"""
    hex_port = f"{port:04X}"
    inodes: set[str] = set()
    for net_file in ("/proc/net/tcp", "/proc/net/tcp6"):
        p = Path(net_file)
        if not p.is_file():
            continue
        try:
            for ln in p.read_text(encoding="utf-8", errors="ignore").splitlines()[1:]:
                cols = ln.split()
                if len(cols) >= 10 and cols[3] == "0A" and cols[1].endswith(f":{hex_port}"):
                    if cols[9] != "0":
                        inodes.add(f"socket:[{cols[9]}]")
        except Exception:
            continue
    return inodes


def cleanup_stale_sgl_omni(port: int = SERVE_PORT) -> None:
    """精准清理残留的 sgl-omni / sgl_venv / 8000 端口监听进程（严禁误伤 Colab/Jupyter 内核），并等待显存释放。"""
    my_pid = os.getpid()
    parent_pid = os.getppid()
    port_inodes = _find_listening_inodes_for_port(port)
    stale_pids: set[int] = set()

    proc_root = Path("/proc")
    if proc_root.is_dir():
        for entry in proc_root.iterdir():
            if not entry.name.isdigit():
                continue
            pid = int(entry.name)
            if pid in (my_pid, parent_pid, 1):
                continue
            try:
                cmdline = (entry / "cmdline").read_bytes().replace(b"\x00", b" ").decode("utf-8", errors="ignore")
                low_cmd = cmdline.lower()
                if any(safe_kw in low_cmd for safe_kw in ("ipykernel", "jupyter", "colab", "dap_multiplexer")):
                    continue

                matched = any(
                    kw in cmdline
                    for kw in ("sgl-omni", "sglang_omni", "/content/sgl_venv/", "sglang::")
                )
                if not matched:
                    try:
                        env_txt = (entry / "environ").read_bytes().decode("utf-8", errors="ignore")
                        if "VIRTUAL_ENV=/content/sgl_venv" in env_txt:
                            matched = True
                    except Exception:
                        pass
                if not matched and port_inodes:
                    fd_dir = entry / "fd"
                    if fd_dir.is_dir():
                        for fd in fd_dir.iterdir():
                            try:
                                if os.readlink(fd) in port_inodes:
                                    matched = True
                                    break
                            except Exception:
                                continue
                if matched:
                    stale_pids.add(pid)
            except Exception:
                continue

    if not stale_pids:
        return

    pid_list = sorted(stale_pids)
    print(f"🧹 检测到残留推理进程 {pid_list}，正在清理以释放 {port} 端口与 GPU 显存...", flush=True)
    for pid in pid_list:
        try:
            os.kill(pid, signal.SIGTERM)
        except Exception:
            pass

    deadline = time.monotonic() + 6.0
    while time.monotonic() < deadline:
        if not any(Path(f"/proc/{p}").exists() for p in pid_list):
            break
        time.sleep(0.3)

    for pid in pid_list:
        if Path(f"/proc/{pid}").exists():
            try:
                os.kill(pid, signal.SIGKILL)
            except Exception:
                pass

    # 等待 NVIDIA 驱动异步回收旧进程占用的大块显存
    if shutil.which("nvidia-smi"):
        vram_deadline = time.monotonic() + 15.0
        while time.monotonic() < vram_deadline:
            try:
                used_str = subprocess.check_output(
                    ["nvidia-smi", "--query-gpu=memory.used", "--format=csv,noheader,nounits"],
                    text=True,
                    stderr=subprocess.DEVNULL,
                ).splitlines()[0].strip()
                if float(used_str) < 2048.0:
                    break
            except Exception:
                break
            time.sleep(1.0)


def stop_server_process(proc: subprocess.Popen | None) -> None:
    """优雅终止后台 sgl-omni 进程组并兜底清理残留子进程。"""
    if proc is not None and proc.poll() is None:
        print("\n🧹 正在关闭后台 sgl-omni 服务并释放显存...", flush=True)
        try:
            os.killpg(os.getpgid(proc.pid), signal.SIGTERM)
        except Exception:
            try:
                proc.terminate()
            except Exception:
                pass
        try:
            proc.wait(timeout=15)
        except subprocess.TimeoutExpired:
            try:
                os.killpg(os.getpgid(proc.pid), signal.SIGKILL)
            except Exception:
                try:
                    proc.kill()
                except Exception:
                    pass
            try:
                proc.wait(timeout=5)
            except Exception:
                pass
    cleanup_stale_sgl_omni(SERVE_PORT)


def check_health(url: str = SGLANG_HEALTH_URL) -> bool:
    try:
        with urllib.request.urlopen(url, timeout=3) as resp:
            return getattr(resp, "status", 200) == 200
    except Exception:
        return False


# =========================
# 2. 读取 API Key（零手动填表）
# =========================
FISH_API_KEY = ""
try:
    from google.colab import userdata  # type: ignore

    FISH_API_KEY = (userdata.get("FISH_API_KEY") or "").strip()
    if FISH_API_KEY:
        print("✅ 已从 Colab Secrets 读取 FISH_API_KEY", flush=True)
except Exception as exc:
    print(f"ℹ️ Colab Secrets 未返回 FISH_API_KEY ({type(exc).__name__})，尝试环境变量/输入兜底...", flush=True)

if not FISH_API_KEY:
    FISH_API_KEY = os.environ.get("FISH_API_KEY", "").strip()
    if FISH_API_KEY:
        print("✅ 已从环境变量读取 FISH_API_KEY", flush=True)

if not FISH_API_KEY:
    try:
        FISH_API_KEY = getpass.getpass("请输入 FISH_API_KEY（输入隐藏）: ").strip()
    except (EOFError, KeyboardInterrupt):
        FISH_API_KEY = ""

if not FISH_API_KEY:
    raise RuntimeError(
        "未获取到 FISH_API_KEY！请在 Colab 左侧 🔑 Secrets 添加 `FISH_API_KEY` 并打开 Notebook 访问开关。"
    )
os.environ["FISH_API_KEY"] = FISH_API_KEY

# 可选读取 HF_TOKEN（若用户在 Secrets 配了则自动加速 Hugging Face 下载，未配绝不报错）
if not os.environ.get("HF_TOKEN"):
    try:
        from google.colab import userdata  # type: ignore

        _hf_tok = (userdata.get("HF_TOKEN") or "").strip()
        if _hf_tok:
            os.environ["HF_TOKEN"] = _hf_tok
            print("✅ 已从 Colab Secrets 读取可选 HF_TOKEN（加速权重拉取）", flush=True)
    except Exception:
        pass


# =========================
# 3. 挂载 Google Drive（读取上一轮合格好句底包 + 保存重抽产物）
# =========================
drive_mounted = False
DRIVE_ROOT = Path("/content/drive/MyDrive")
try:
    from google.colab import drive  # type: ignore

    if not DRIVE_ROOT.is_dir():
        drive.mount("/content/drive", force_remount=False)
    drive_mounted = DRIVE_ROOT.is_dir()
except Exception as exc:
    print(f"⚠️ Google Drive 挂载未启用或失败 ({exc})，将回退使用本地目录保存产物。", flush=True)

if drive_mounted:
    WORK_DIR = DRIVE_ROOT / "ojousama_voice_work"
    DRIVE_ZIP_ROOT_COPY = DRIVE_ROOT / "ojousama_jp_voice_pack.zip"
    print(f"✅ Google Drive 已挂载，断点与成品持久化目录: {WORK_DIR}", flush=True)
else:
    WORK_DIR = Path("/content/ojousama_voice_work_fallback")
    DRIVE_ZIP_ROOT_COPY = None
    print(f"ℹ️ 当前使用本地工作目录: {WORK_DIR}", flush=True)

MP3_OUT_DIR = WORK_DIR / "mp3_out"
CACHE_JSON = WORK_DIR / ".synth_cache.json"
MP3_OUT_DIR.mkdir(parents=True, exist_ok=True)


# =========================
# 4. 自动识别 Colab 显卡、同卡显存配额与 ASR 设备
# =========================
def detect_gpu_spec() -> dict[str, object]:
    try:
        out = subprocess.check_output(
            ["nvidia-smi", "--query-gpu=name,memory.total,compute_cap", "--format=csv,noheader,nounits"],
            text=True,
            stderr=subprocess.DEVNULL,
        ).strip()
        if out:
            first = out.splitlines()[0]
            name, mem_mib, cc_str = [x.strip() for x in first.rsplit(",", 2)]
            major, minor = [int(x) for x in cc_str.split(".")[:2]]
            return {
                "has_gpu": True,
                "name": name,
                "vram_gb": round(float(mem_mib) / 1024.0, 1),
                "cc": (major, minor),
            }
    except Exception:
        pass
    try:
        import torch  # type: ignore

        if torch.cuda.is_available():
            props = torch.cuda.get_device_properties(0)
            return {
                "has_gpu": True,
                "name": props.name,
                "vram_gb": round(props.total_memory / (1024 ** 3), 1),
                "cc": (props.major, props.minor),
            }
    except Exception:
        pass
    return {"has_gpu": False, "name": "CPU", "vram_gb": 0.0, "cc": (0, 0)}


gpu_info = detect_gpu_spec()
has_gpu = bool(gpu_info["has_gpu"])
gpu_name = str(gpu_info["name"])
vram_gb = float(gpu_info["vram_gb"])
cc: tuple[int, int] = gpu_info["cc"]  # type: ignore[assignment]
sm_tag = f"sm_{cc[0]}{cc[1]}" if has_gpu else "cpu"

extra_serve_args: list[str] = []
is_40gb_tight = False

if has_gpu and cc >= (8, 9) and vram_gb >= 64.0:
    # G4 (RTX PRO 6000 Blackwell 96GB, sm_120) / H100 / H200 / B200：
    # 限制 tts_engine 显存占比 0.62，留足充裕显存给同卡 FP16 anime-whisper 实时质检！
    BACKEND = "sglang"
    MODEL_ID = "fishaudio/drama-3"
    CONCURRENCY = 2
    ASR_DEVICE = "cuda"
    extra_serve_args = [
        "--tts_engine.gpu_memory_fraction", "0.62",
    ]
    TIER_DESC = (
        f"{gpu_name} ({vram_gb} GB, {sm_tag}) -> 满血原版 FP8 ({MODEL_ID}), "
        f"tts_mem=0.62 + 同卡 ASR-QC ({ASR_DEVICE}), concurrency={CONCURRENCY}"
    )
elif has_gpu and cc >= (8, 0) and vram_gb >= 64.0:
    # A100 80GB (sm_80)：无原生 FP8 走 int4，限制 tts_engine 0.62 供同卡 FP16 anime-whisper 共存！
    BACKEND = "sglang"
    MODEL_ID = "fishaudio/drama-3-int4"
    CONCURRENCY = 2
    ASR_DEVICE = "cuda"
    extra_serve_args = [
        "--tts_engine.gpu_memory_fraction", "0.62",
    ]
    TIER_DESC = (
        f"{gpu_name} ({vram_gb} GB, {sm_tag}) -> INT4 ({MODEL_ID}), "
        f"tts_mem=0.62 + 同卡 ASR-QC ({ASR_DEVICE}), concurrency={CONCURRENCY}"
    )
elif has_gpu and cc >= (8, 0) and vram_gb >= 36.0:
    # A100 40GB (sm_80)：5 项显存收紧参数 + ASR 走 CPU 稳妥不抢显存！
    BACKEND = "sglang"
    MODEL_ID = "fishaudio/drama-3-int4"
    CONCURRENCY = 1
    ASR_DEVICE = "cpu"
    is_40gb_tight = True
    extra_serve_args = [
        "--tts_engine.gpu_memory_fraction", "0.58",
        "--vocoder.gpu_memory_fraction", "0.12",
        "--tts_engine.factory.max_new_tokens", "1024",
        "--preprocessing.total_reserve_bytes", "3GiB",
        "--preprocessing.factory.max_reference_audio_seconds", "20",
    ]
    TIER_DESC = (
        f"{gpu_name} ({vram_gb} GB, {sm_tag}) -> INT4 显存收紧模式 ({MODEL_ID}), "
        f"ASR-QC ({ASR_DEVICE}), concurrency={CONCURRENCY}"
    )
else:
    # L4 / T4 / CPU：免启本地 sgl-omni，直连 Fish Audio Cloud API，本地显卡专供 anime-whisper 质检
    BACKEND = "fish_api"
    MODEL_ID = FISH_API_FALLBACK_MODEL
    CONCURRENCY = 2
    ASR_DEVICE = "cuda" if has_gpu else "cpu"
    TIER_DESC = (
        f"{gpu_name} ({vram_gb} GB, {sm_tag}) -> 直连 Fish Audio API ({FISH_API_FALLBACK_MODEL}) "
        f"+ 本地 ASR-QC ({ASR_DEVICE})"
    )

print(f"\n🖥️ 显卡自动识别结果: {TIER_DESC}", flush=True)


# =========================
# 5. 安装系统依赖、宿主 --asr-qc 依赖与预缓存 litagin/anime-whisper
# =========================
def get_node_major() -> int:
    node_bin = shutil.which("node")
    if not node_bin:
        return 0
    try:
        ver = subprocess.check_output([node_bin, "-v"], text=True, stderr=subprocess.DEVNULL).strip()
        return int(ver.lstrip("vV").split(".")[0])
    except Exception:
        return 0


# Colab 常在 /tools/node/bin 预置 Node 20，若未进 PATH 先补齐
if get_node_major() < 18 and Path("/tools/node/bin/node").is_file():
    os.environ["PATH"] = f"/tools/node/bin:{os.environ.get('PATH', '')}"

apt_env = {**os.environ, "DEBIAN_FRONTEND": "noninteractive"}
apt_needed: list[str] = []
if not shutil.which("ffmpeg"):
    apt_needed.append("ffmpeg")
if not shutil.which("sox"):
    apt_needed.extend(["sox", "libsox-fmt-all"])

if apt_needed or get_node_major() < 18:
    run_stream(["apt-get", "update", "-qq"], env=apt_env)
    if apt_needed:
        run_stream(["apt-get", "install", "-y", "-qq", *apt_needed], env=apt_env)
    if get_node_major() < 18:
        print("📦 正在安装 Node.js 20 LTS (>= 18)...", flush=True)
        run_stream(["apt-get", "install", "-y", "-qq", "ca-certificates", "curl", "gnupg"], env=apt_env)
        run_stream(
            ["bash", "-lc", "curl -fsSL https://deb.nodesource.com/setup_20.x | bash - && apt-get install -y -qq nodejs"],
            env=apt_env,
        )

assert shutil.which("ffmpeg"), "ffmpeg 安装失败！"
assert shutil.which("sox"), "sox 安装失败！"
assert get_node_major() >= 18, f"Node.js 版本必须 >= 18，当前检测到 major={get_node_major()}"
print(f"✅ 系统依赖就绪: ffmpeg, sox, node ({subprocess.check_output(['node', '-v'], text=True).strip()})", flush=True)

# 补齐宿主 Python 工具与 --asr-qc 质检依赖（Colab 已预装 torch/transformers，仅秒级补装缺失项）
host_pip_pkgs: list[str] = []
if not shutil.which("uv") and importlib.util.find_spec("uv") is None:
    host_pip_pkgs.append("uv")
if importlib.util.find_spec("requests") is None:
    host_pip_pkgs.append("requests")
if importlib.util.find_spec("huggingface_hub") is None:
    host_pip_pkgs.append("huggingface_hub>=0.26.0")
for _asr_pkg in ("transformers", "accelerate", "pykakasi", "torchaudio"):
    if importlib.util.find_spec(_asr_pkg) is None:
        host_pip_pkgs.append(_asr_pkg)

# 确保 safetensors >= 0.4.3（支持 FP8 F8_E4M3 分片头解析）
_need_st = importlib.util.find_spec("safetensors") is None
if not _need_st:
    try:
        import safetensors as _st

        _ver_parts = [int(x) for x in _st.__version__.split(".")[:2] if x.isdigit()]
        if tuple(_ver_parts) < (0, 4):
            _need_st = True
    except Exception:
        _need_st = True
if _need_st:
    host_pip_pkgs.append("safetensors>=0.4.3")

if host_pip_pkgs:
    print(f"📦 正在为宿主 Python 补装依赖: {', '.join(host_pip_pkgs)} ...", flush=True)
    run_stream([sys.executable, "-m", "pip", "install", "-q", *host_pip_pkgs])


def precache_anime_whisper(repo_id: str = ANIME_WHISPER_REPO_ID, max_attempts: int = 3) -> None:
    """提前将 litagin/anime-whisper (~1.5GB) 缓存到本地 NVMe SSD (/content/hf_cache)。"""
    from huggingface_hub import snapshot_download

    assert not str(HF_HOME_DIR.resolve()).startswith("/content/drive"), "HF_HOME 严禁指向 Google Drive！"
    for attempt in range(1, max_attempts + 1):
        try:
            print(
                f"📥 [ASR 质检模型预热 {attempt}/{max_attempts}] 缓存 {repo_id} -> {HF_HOME_DIR} ...",
                flush=True,
            )
            cached_dir = snapshot_download(
                repo_id=repo_id,
                cache_dir=str(HF_HOME_DIR / "hub"),
                ignore_patterns=["*.msgpack", "*.h5", "flax_model*"],
                max_workers=4,
            )
            print(f"✅ ASR 质检模型已就绪: {cached_dir}", flush=True)
            return
        except Exception as exc:
            if attempt == max_attempts:
                raise RuntimeError(f"预下载 {repo_id} 连续 {max_attempts} 次失败: {exc}") from exc
            print(f"⚠️ 预下载 {repo_id} 网络波动 ({exc})，3 秒后自动重试...", flush=True)
            time.sleep(3.0)


precache_anime_whisper()


def resolve_uv_bin() -> str:
    found = shutil.which("uv")
    if found:
        return found
    out = subprocess.check_output(
        [sys.executable, "-c", "import uv; print(uv.find_uv_bin())"],
        text=True,
    ).strip()
    if out and Path(out).is_file():
        return out
    raise RuntimeError("未能定位 uv 可执行文件！")


# 克隆或更新业务仓库到 /content/fictional-adventure（确保拉到最新升级版 batch_synthesize.py / casting / retake_bad_ids.txt）
if not (REPO_DIR / ".git").is_dir():
    if REPO_DIR.exists():
        shutil.rmtree(REPO_DIR)
    run_stream(["git", "clone", "--depth", "1", "--branch", "main", REPO_URL, str(REPO_DIR)])
else:
    run_stream(["git", "fetch", "origin", "main", "--depth", "1"], cwd=REPO_DIR)
    run_stream(["git", "checkout", "main"], cwd=REPO_DIR)
    run_stream(["git", "reset", "--hard", "origin/main"], cwd=REPO_DIR)


# =========================
# 6. 独立虚拟环境安装 sglang-omni[fishaudio] 与本地 SSD 权重校验
# =========================
def ensure_sglang_omni_venv() -> Path:
    """【防崩红线 #2】在独立 /content/sgl_venv 中配合 protobuf overrides 安装 sglang-omni[fishaudio]。"""
    sgl_bin = SGL_VENV_DIR / "bin" / "sgl-omni"
    venv_py = SGL_VENV_DIR / "bin" / "python"

    if sgl_bin.is_file() and venv_py.is_file():
        probe = subprocess.run(
            [str(venv_py), "-c", "import sglang_omni, dac, audiotools"],
            capture_output=True,
            text=True,
        )
        if probe.returncode == 0:
            print("✅ /content/sgl_venv/bin/sgl-omni 已存在且自检通过，直接复用跳过重装！", flush=True)
            return sgl_bin
        print("⚠️ 检测到旧 /content/sgl_venv 依赖不完整，将重建独立虚拟环境...", flush=True)

    uv_bin = resolve_uv_bin()

    if not ((SGL_SRC_DIR / ".git").is_dir() and (SGL_SRC_DIR / "pyproject.toml").is_file()):
        if SGL_SRC_DIR.exists():
            shutil.rmtree(SGL_SRC_DIR)
        run_stream(
            ["git", "clone", "--depth", "1", "https://github.com/sgl-project/sglang-omni.git", str(SGL_SRC_DIR)]
        )

    pyproject = tomllib.loads((SGL_SRC_DIR / "pyproject.toml").read_text(encoding="utf-8"))
    overrides = pyproject.get("tool", {}).get("uv", {}).get("override-dependencies") or [
        "protobuf>=6.31.1,<7.0.0; sys_platform != 'darwin' or platform_machine != 'arm64'",
    ]
    SGL_OVERRIDES_PATH.write_text("\n".join(overrides) + "\n", encoding="utf-8")
    print(f"✅ 已写出 /content/omni-overrides.txt ({len(overrides)} 条 override 规则)", flush=True)

    if SGL_VENV_DIR.exists():
        shutil.rmtree(SGL_VENV_DIR)
    uv_env = {**os.environ, "UV_HTTP_TIMEOUT": "300", "UV_HTTP_RETRIES": "5"}
    run_stream([uv_bin, "venv", str(SGL_VENV_DIR), "--python", "3.12"], env=uv_env)
    # 在 SGL_SRC_DIR 下执行以自动生效 pyproject.toml 中的 [tool.uv] environments 平台约束
    run_stream(
        [
            uv_bin, "pip", "install",
            "--python", str(venv_py),
            "--prerelease=allow",
            "--overrides", str(SGL_OVERRIDES_PATH),
            "-e", f"{SGL_SRC_DIR}[fishaudio]",
        ],
        cwd=SGL_SRC_DIR,
        env=uv_env,
    )

    if not sgl_bin.is_file():
        raise RuntimeError(f"sglang-omni 安装后未找到可执行文件: {sgl_bin}")
    run_stream([str(venv_py), "-c", "import sglang_omni, dac, audiotools; print('✅ sglang_omni + dac + audiotools 自检通过')"])
    return sgl_bin


def _unlink_with_target(path: Path) -> None:
    """若文件是符号链接，连同底层真实缓存 blob 一并删除，防止重下时命中损坏缓存。"""
    try:
        real_target = path.resolve() if path.is_symlink() else None
    except Exception:
        real_target = None
    path.unlink(missing_ok=True)
    if real_target is not None and real_target != path and real_target.is_file():
        real_target.unlink(missing_ok=True)


def download_and_verify_local_model(repo_id: str, max_attempts: int = 3) -> Path:
    """【防崩红线 #1】下载模型到本地 NVMe SSD 并用 safetensors.safe_open 逐片校验分片头与完整性。"""
    from huggingface_hub import hf_hub_download, snapshot_download
    from safetensors import safe_open

    local_dir = LOCAL_MODELS_ROOT / repo_id.replace("/", "--")
    assert not str(local_dir.resolve()).startswith("/content/drive"), "模型目录严禁位于 Google Drive！"
    local_dir.mkdir(parents=True, exist_ok=True)

    for attempt in range(1, max_attempts + 1):
        print(f"\n📥 [权重准备 {attempt}/{max_attempts}] 正在下载/核对 {repo_id} -> 本地高速盘 {local_dir} ...", flush=True)
        dl_error: str | None = None
        try:
            snapshot_download(
                repo_id=repo_id,
                local_dir=str(local_dir),
                ignore_patterns=["eval/*", "eval_ab/*", "media/*", "samples/*", "*.mp4", "*.png"],
                max_workers=4,
            )
        except Exception as exc:
            dl_error = str(exc)
            print(f"⚠️ 第 {attempt} 轮 snapshot_download 网络中断或异常: {dl_error}", flush=True)

        # 检查核心元数据文件、残留 .incomplete 与分片完整性
        missing_meta = [
            fn for fn in ("config.json", "tokenizer.json") if not (local_dir / fn).is_file()
        ]
        incompletes = list(local_dir.rglob("*.incomplete")) + list(HF_HOME_DIR.rglob("*.incomplete"))
        shards = sorted(p for p in local_dir.rglob("*.safetensors") if ".cache" not in p.parts)

        missing_shards: list[Path] = []
        for idx_file in (p for p in local_dir.rglob("*.safetensors.index.json") if ".cache" not in p.parts):
            try:
                idx_data = json.loads(idx_file.read_text(encoding="utf-8"))
                expected_names = sorted(set((idx_data.get("weight_map") or {}).values()))
                for rel_name in expected_names:
                    cand = idx_file.parent / rel_name
                    if not cand.is_file():
                        missing_shards.append(cand)
            except Exception as exc:
                missing_meta.append(f"{idx_file.name}({exc})")

        bad_shards: list[tuple[Path, str]] = []
        for shard in shards:
            try:
                if shard.stat().st_size < 64:
                    bad_shards.append((shard, f"文件过小 ({shard.stat().st_size} B)"))
                    continue
                with safe_open(str(shard), framework="np") as sf:
                    if not sf.keys():
                        bad_shards.append((shard, "分片头不包含任何张量键"))
            except Exception as exc:
                bad_shards.append((shard, str(exc)))

        if (
            dl_error is None
            and not missing_meta
            and not incompletes
            and shards
            and not missing_shards
            and not bad_shards
        ):
            total_gib = sum(s.stat().st_size for s in shards) / (1024 ** 3)
            print(
                f"✅ 本地 SSD 模型校验通过：{len(shards)} 个 .safetensors 分片全部合法（共 {total_gib:.2f} GiB）！",
                flush=True,
            )
            break

        print(
            f"⚠️ 权重校验未通过 (dl_err={bool(dl_error)}, missing_meta={missing_meta}, "
            f"incomplete={len(incompletes)}, missing_shards={len(missing_shards)}, bad_shards={len(bad_shards)})，"
            "正在清理损坏文件并自动重试...",
            flush=True,
        )
        for inc in incompletes:
            print(f"  🗑️ 删除残留 .incomplete: {inc}", flush=True)
            _unlink_with_target(inc)
        for shard, reason in bad_shards:
            print(f"  🗑️ 删除损坏分片 {shard.name}: {reason}", flush=True)
            _unlink_with_target(shard)
        if bad_shards or missing_shards or missing_meta:
            shutil.rmtree(local_dir / ".cache", ignore_errors=True)

        if attempt == max_attempts:
            raise RuntimeError(
                f"模型 {repo_id} 连续 {max_attempts} 次下载校验失败："
                f"dl_error={dl_error}, missing_meta={missing_meta}, shards={len(shards)}, "
                f"missing_shards={len(missing_shards)}, bad_shards={len(bad_shards)}, incomplete={len(incompletes)}"
            )
        time.sleep(3.0)

    # 预热 Drama 3 默认引用的 S2-Pro codec.pth 到本地 HF_HOME，避免 sgl-omni 启动阶段额外联网等待
    try:
        hf_hub_download(
            repo_id="fishaudio/s2-pro",
            filename="codec.pth",
            revision="1de9996b6be38b745688de084d87a5633f714e4e",
        )
    except Exception:
        pass

    return local_dir


def start_sgl_omni_and_wait_ready(
    sgl_omni_bin: Path,
    local_model_dir: Path,
) -> tuple[subprocess.Popen, IO[str]]:
    """清理旧进程、启动后台 sgl-omni serve 并流式轮询 /health；若提前退出则打印最后 150 行完整日志。"""
    cleanup_stale_sgl_omni(SERVE_PORT)

    serve_env = os.environ.copy()
    serve_env["PYTHONUNBUFFERED"] = "1"
    serve_env["HF_HOME"] = str(HF_HOME_DIR)
    serve_env["HUGGINGFACE_HUB_CACHE"] = str(HF_HOME_DIR / "hub")
    serve_env["VIRTUAL_ENV"] = str(SGL_VENV_DIR)
    serve_env["PATH"] = f"{SGL_VENV_DIR / 'bin'}:{serve_env.get('PATH', '')}"
    # 对齐 sglang-omni 内部 stage 启动/预热超时与外层轮询上限，并禁止端口被占时静默漂移到其他端口
    serve_env["SGLANG_OMNI_STARTUP_TIMEOUT"] = str(STARTUP_TIMEOUT_SEC)
    serve_env["SGLANG_OMNI_WARMUP_TIMEOUT"] = str(STARTUP_TIMEOUT_SEC)
    serve_env["SGLANG_OMNI_STRICT_PORT"] = "1"

    serve_cmd = [
        str(sgl_omni_bin), "serve",
        "--model-path", str(local_model_dir),
        "--host", SERVE_HOST,
        "--port", str(SERVE_PORT),
        "--allowed-local-media-path", str(LOCAL_REFS_DIR.resolve()),
        *extra_serve_args,
    ]
    print(f"\n🚀 启动本地 SGLang-Omni Drama 3 服务:\n$ {mask_cmd(serve_cmd)}", flush=True)
    log_fp = SERVE_LOG_PATH.open("w", encoding="utf-8", buffering=1)
    proc = subprocess.Popen(
        serve_cmd,
        cwd=str(REPO_DIR),
        env=serve_env,
        stdout=log_fp,
        stderr=subprocess.STDOUT,
        start_new_session=True,
    )

    t_boot = time.monotonic()
    log_offset = 0
    while True:
        if SERVE_LOG_PATH.is_file():
            with SERVE_LOG_PATH.open("rb") as lf:
                lf.seek(log_offset)
                raw_chunk = lf.read()
                log_offset = lf.tell()
            if raw_chunk:
                txt_chunk = raw_chunk.decode("utf-8", errors="replace")
                print(txt_chunk, end="" if txt_chunk.endswith("\n") else "\n", flush=True)

        rc = proc.poll()
        if rc is not None:
            time.sleep(0.5)
            log_fp.flush()
            tail_150 = read_log_tail(SERVE_LOG_PATH, 150)
            print(
                f"\n❌ sgl-omni 进程提前异常退出 (exit={rc})！以下为日志文件最后 150 行：\n"
                + "=" * 80
                + f"\n{tail_150}\n"
                + "=" * 80,
                flush=True,
            )
            log_fp.close()
            raise RuntimeError(f"sgl-omni serve 启动失败 (exit={rc})，详见上方最后 150 行完整异常日志。")

        if check_health(SGLANG_HEALTH_URL):
            elapsed_boot = int(time.monotonic() - t_boot)
            print(f"\n✅ sgl-omni 服务已就绪 ({SGLANG_HEALTH_URL}, 启动耗时 {elapsed_boot}s)！", flush=True)
            return proc, log_fp

        if time.monotonic() - t_boot > STARTUP_TIMEOUT_SEC:
            tail_150 = read_log_tail(SERVE_LOG_PATH, 150)
            print(
                f"\n❌ 等待 sgl-omni /health 超时 ({STARTUP_TIMEOUT_SEC}s)！以下为日志文件最后 150 行：\n"
                + "=" * 80
                + f"\n{tail_150}\n"
                + "=" * 80,
                flush=True,
            )
            stop_server_process(proc)
            log_fp.close()
            raise TimeoutError(f"等待 sgl-omni /health 超过 {STARTUP_TIMEOUT_SEC}s 超时。")

        time.sleep(2.0)


# =========================
# 7. 智能判定「增量重抽坏句」vs「全新全量生产」
# =========================
master_raw = json.loads((REPO_DIR / "tools/voice/script_jp_master.json").read_text(encoding="utf-8"))
master_items = (
    master_raw
    if isinstance(master_raw, list)
    else (master_raw.get("entries") or master_raw.get("items") or master_raw.get("lines") or [])
)

id_to_rel_mp3: dict[str, str] = {}
valid_rel_paths: set[str] = set()
for _item in master_items:
    if not isinstance(_item, dict):
        continue
    _vid = str(_item.get("voiceId") or "").strip()
    _rel = str(_item.get("audioRelPath") or _item.get("audioPath") or "").replace("\\", "/").lstrip("/")
    if _rel.startswith("assets/voice/"):
        _rel = _rel[len("assets/voice/") :]
    if _vid and _rel:
        id_to_rel_mp3[_vid] = _rel
        valid_rel_paths.add(_rel)

retake_list_repo_path = REPO_DIR / "tools/voice/retake_bad_ids.txt"
bad_id_set: set[str] = set()
if retake_list_repo_path.is_file():
    bad_id_set = {
        ln.strip()
        for ln in retake_list_repo_path.read_text(encoding="utf-8").splitlines()
        if ln.strip() and not ln.strip().startswith("#")
    }

existing_good_ids: set[str] = set()
missing_good_ids: list[str] = []
for _vid, _rel in id_to_rel_mp3.items():
    if _vid in bad_id_set:
        continue
    _mp3_file = MP3_OUT_DIR / _rel
    if _mp3_file.is_file() and _mp3_file.stat().st_size >= 1024:
        existing_good_ids.add(_vid)
    else:
        missing_good_ids.append(_vid)

incremental_retake_mode = bool(bad_id_set) and (len(existing_good_ids) >= GOOD_BASE_PACK_MIN_COUNT)
if incremental_retake_mode:
    if missing_good_ids:
        # 若云盘中偶有少量非坏句缺失，自动并入本次重抽清单，确保最终 1,276 条 100% 齐全且绝不重跑已有好句
        ordered_targets = [
            _vid for _vid in id_to_rel_mp3 if (_vid in bad_id_set or _vid in set(missing_good_ids))
        ]
        EFFECTIVE_RETAKE_IDS_PATH.write_text("\n".join(ordered_targets) + "\n", encoding="utf-8")
        only_ids_arg = f"@{EFFECTIVE_RETAKE_IDS_PATH}"
    else:
        ordered_targets = [_vid for _vid in id_to_rel_mp3 if _vid in bad_id_set]
        only_ids_arg = "@tools/voice/retake_bad_ids.txt"

    MAX_RETRIES = 5
    MODE_BANNER = (
        f"🎯 智能增量重抽模式：检测到 {len(existing_good_ids)} 条已合格原装好句（100% 保留不重跑），"
        f"仅精准重抽 {len(ordered_targets)} 条目标句 (--only-ids {only_ids_arg} --force --asr-qc --max-retries {MAX_RETRIES})"
    )
else:
    only_ids_arg = ""
    MAX_RETRIES = 4
    MODE_BANNER = (
        f"🆕 全量合成模式（当前底包好句数={len(existing_good_ids)} < {GOOD_BASE_PACK_MIN_COUNT}）："
        f"自动回退执行全量 {len(id_to_rel_mp3)} 条合成 (--scope all --asr-qc --max-retries {MAX_RETRIES})"
    )

print(f"\n{MODE_BANNER}", flush=True)


# =========================
# 8. 主执行流水线（刷新参考音 -> 启服务 -> 质检重抽 -> 合并 1,276 条全量打包）
# =========================
serve_proc: subprocess.Popen | None = None
serve_log_fp: IO[str] | None = None
sgl_omni_bin: Path | None = None
local_model_dir: Path | None = None
QA_REPORT_PATH = REPO_DIR / "tools/voice/qa_report.json"


def build_synth_cmd(*, override_only_ids: str | None = None) -> list[str]:
    use_only_ids = only_ids_arg if override_only_ids is None else override_only_ids
    if BACKEND == "sglang":
        cmd = [
            sys.executable, "tools/voice/batch_synthesize.py",
            "--script", "tools/voice/script_jp_master.json",
            "--refs", str(REFS_JSON),
            "--casting", "tools/voice/casting_fish_audio.json",
            "--out-dir", str(MP3_OUT_DIR),
            "--cache-file", str(CACHE_JSON),
            "--qa-report", "tools/voice/qa_report.json",
            "--backend", "sglang",
            "--sglang-url", SGLANG_SPEECH_URL,
            "--model-id", MODEL_ID,
            "--concurrency", str(CONCURRENCY),
            "--max-ref-clips", "1" if is_40gb_tight else "2",
            "--asr-qc",
            "--asr-device", ASR_DEVICE,
            "--max-retries", str(MAX_RETRIES),
            "--skip-manifest",
        ]
    else:
        cmd = [
            sys.executable, "tools/voice/batch_synthesize.py",
            "--script", "tools/voice/script_jp_master.json",
            "--casting", "tools/voice/casting_fish_audio.json",
            "--out-dir", str(MP3_OUT_DIR),
            "--cache-file", str(CACHE_JSON),
            "--qa-report", "tools/voice/qa_report.json",
            "--backend", "fish_api",
            "--fish-api-model", FISH_API_FALLBACK_MODEL,
            "--fish-api-key", FISH_API_KEY,
            "--concurrency", str(CONCURRENCY),
            "--asr-qc",
            "--asr-device", ASR_DEVICE,
            "--max-retries", str(MAX_RETRIES),
            "--skip-manifest",
        ]

    if use_only_ids:
        cmd += ["--only-ids", use_only_ids, "--force"]
    else:
        cmd += ["--scope", "all"]
    return cmd


try:
    if BACKEND == "sglang":
        sgl_omni_bin = ensure_sglang_omni_venv()

        # 8.1 拉取角色参考音到本地 /content/refs_wav（若其它 24 套缓存完好，仅强制刷新升级后的 npc_child_f！）
        can_incremental_refs = False
        if REFS_JSON.is_file():
            try:
                _old_profiles = json.loads(REFS_JSON.read_text(encoding="utf-8")).get("profiles") or {}
                _valid_others = sum(
                    1
                    for _pid, _p in _old_profiles.items()
                    if _pid != "npc_child_f"
                    and isinstance(_p, dict)
                    and _p.get("references")
                    and all(Path(_r.get("audio_path", "")).is_file() for _r in _p["references"])
                )
                can_incremental_refs = _valid_others >= 24
            except Exception:
                can_incremental_refs = False

        fetch_cmd = [
            sys.executable, "tools/voice/fetch_fish_refs.py",
            "--casting", "tools/voice/casting_fish_audio.json",
            "--out", str(REFS_JSON),
            "--refs-dir", str(LOCAL_REFS_DIR),
            "--api-key", FISH_API_KEY,
        ]
        if is_40gb_tight:
            fetch_cmd += ["--max-seconds", "19", "--max-clips", "1"]
        if can_incremental_refs:
            fetch_cmd += ["--only-profiles", "npc_child_f", "--force"]
            print("\n🎙️ [步骤 1/4] 复用 24 套已解析参考音，定向强制刷新 npc_child_f (幼い子供の声)...", flush=True)
        else:
            print("\n🎙️ [步骤 1/4] 全量拉取 Fish Audio 25 角色纯日语参考音（含新版 npc_child_f）到本地 SSD...", flush=True)

        fetch_rc = run_stream(fetch_cmd, cwd=REPO_DIR, check=False)
        if not REFS_JSON.is_file():
            raise RuntimeError("fetch_fish_refs.py 未生成 refs_resolved.json，请检查 FISH_API_KEY 与网络！")
        ref_summary = json.loads(REFS_JSON.read_text(encoding="utf-8")).get("summary", {})
        if int(ref_summary.get("resolved", 0)) <= 0:
            raise RuntimeError(f"25 套角色参考音全部拉取失败: {ref_summary}")
        if fetch_rc != 0:
            print(f"⚠️ 参考音拉取存在部分降级/告警 ({ref_summary})，继续合成已就绪角色。", flush=True)

        # 8.2 下载并校验 Drama 3 模型到本地 NVMe SSD
        local_model_dir = download_and_verify_local_model(MODEL_ID)

        # 8.3 启动 sgl-omni serve 并等待 /health 就绪
        print("\n🚀 [步骤 2/4] 启动本地推理引擎并等待健康检查就绪...", flush=True)
        serve_proc, serve_log_fp = start_sgl_omni_and_wait_ready(sgl_omni_bin, local_model_dir)

    synth_cmd = build_synth_cmd()
    QA_REPORT_PATH.unlink(missing_ok=True)
    print(f"\n🎬 [步骤 3/4] 启动日语语音合成 + 同卡 anime-whisper 质检与响度归一化 (backend={BACKEND})...", flush=True)
    synth_rc = run_stream(synth_cmd, cwd=REPO_DIR, check=False)

    # 若偶发个别条目失败，先核查服务存活（若挂掉自动重启服务），再仅针对 failedLines 精准补抽（绝不重复重跑已成功句）
    if synth_rc != 0 and QA_REPORT_PATH.is_file():
        qa_mid_doc = json.loads(QA_REPORT_PATH.read_text(encoding="utf-8"))
        qa_mid = qa_mid_doc.get("stats", {})
        failed_retry_ids = [
            str(x["voiceId"]).strip()
            for x in (qa_mid_doc.get("failedLines") or [])
            if isinstance(x, dict) and x.get("voiceId")
        ]
        if int(qa_mid.get("failed", 0)) > 0 and int(qa_mid.get("succeeded", 0)) > 0 and failed_retry_ids:
            print(
                f"\n🔁 首轮完成 {qa_mid['succeeded']} 条，偶发失败 {len(failed_retry_ids)} 条，正在针对失败条目精准补抽...",
                flush=True,
            )
            if BACKEND == "sglang" and (
                serve_proc is None or serve_proc.poll() is not None or not check_health(SGLANG_HEALTH_URL)
            ):
                print(
                    "⚠️ 检测到 sgl-omni 服务在批产中途退出，最后 150 行日志如下：\n"
                    + "=" * 80
                    + f"\n{read_log_tail(SERVE_LOG_PATH, 150)}\n"
                    + "=" * 80,
                    flush=True,
                )
                stop_server_process(serve_proc)
                if serve_log_fp is not None:
                    serve_log_fp.close()
                    serve_log_fp = None
                assert sgl_omni_bin is not None and local_model_dir is not None
                serve_proc, serve_log_fp = start_sgl_omni_and_wait_ready(sgl_omni_bin, local_model_dir)

            RETRY_FAILED_IDS_PATH.write_text("\n".join(failed_retry_ids) + "\n", encoding="utf-8")
            retry_cmd = build_synth_cmd(override_only_ids=f"@{RETRY_FAILED_IDS_PATH}")
            synth_rc = run_stream(retry_cmd, cwd=REPO_DIR, check=False)

    if not QA_REPORT_PATH.is_file():
        if BACKEND == "sglang" and SERVE_LOG_PATH.is_file():
            print(
                "\n--- sgl-omni 最后 150 行日志 ---\n" + read_log_tail(SERVE_LOG_PATH, 150),
                flush=True,
            )
        raise RuntimeError(f"batch_synthesize.py 异常中止 (exit={synth_rc}) 且未生成 qa_report.json！")

    qa_stats = json.loads(QA_REPORT_PATH.read_text(encoding="utf-8")).get("stats", {})
    if int(qa_stats.get("succeeded", 0)) <= 0:
        raise RuntimeError(f"批量合成成功条数为 0，请检查上方日志: {qa_stats}")
    if synth_rc != 0:
        print(
            f"⚠️ 补抽结束后仍有 {qa_stats.get('failed')} 条失败（详见 tools/voice/qa_report.json），继续合并打包可用语音。",
            flush=True,
        )

    # 合成完毕后立即关闭后台 sgl-omni 进程并释放显存（finally 中保留幂等保护）
    if serve_proc is not None:
        stop_server_process(serve_proc)
        serve_proc = None
    if serve_log_fp is not None:
        serve_log_fp.close()
        serve_log_fp = None

    # =========================
    # 9. 全量合并 1,276 个 MP3、构建 Manifest、打包备份并触发浏览器下载
    # =========================
    print("\n📦 [步骤 4/4] 合并原装好句与重抽修复句至 assets/voice/，构建清单并打包 ojousama_jp_voice_pack.zip ...", flush=True)
    repo_voice_dir = REPO_DIR / "assets/voice"
    shutil.rmtree(repo_voice_dir, ignore_errors=True)
    repo_voice_dir.mkdir(parents=True, exist_ok=True)

    synced_mp3_count = 0
    for src_mp3 in sorted(MP3_OUT_DIR.rglob("*.mp3")):
        if not src_mp3.is_file() or src_mp3.name.endswith(".tmp.mp3") or src_mp3.stat().st_size <= 0:
            continue
        rel_posix = src_mp3.relative_to(MP3_OUT_DIR).as_posix()
        if valid_rel_paths and rel_posix not in valid_rel_paths:
            continue
        dst_mp3 = repo_voice_dir / rel_posix
        dst_mp3.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(src_mp3, dst_mp3)
        synced_mp3_count += 1
    print(
        f"✅ 已全量同步 {synced_mp3_count} / {len(valid_rel_paths)} 个有效 MP3 文件至 {repo_voice_dir} "
        f"（保留原装好句 {len(existing_good_ids)} 条 + 本轮新产出修复句）",
        flush=True,
    )

    # 注意：build_manifest.mjs 手写参数解析，必须用空格分隔 `--voice-dir assets/voice`，严禁用等号！
    run_stream(["node", "tools/voice/build_manifest.mjs", "--voice-dir", "assets/voice"], cwd=REPO_DIR)

    js_manifest_path = REPO_DIR / "js/voice-manifest.js"
    json_manifest_path = REPO_DIR / "assets/voice/manifest.json"
    for req_file in (js_manifest_path, json_manifest_path, QA_REPORT_PATH):
        if not req_file.is_file():
            raise FileNotFoundError(f"打包前缺失关键产物文件: {req_file}")

    # 保持相对仓库根目录结构打包为 /content/ojousama_jp_voice_pack.zip
    LOCAL_ZIP_PATH.unlink(missing_ok=True)
    archived_files = 0
    with zipfile.ZipFile(LOCAL_ZIP_PATH, "w", compression=zipfile.ZIP_DEFLATED) as zf:
        for p in sorted(repo_voice_dir.rglob("*")):
            if not p.is_file() or p.name.endswith(".tmp.mp3"):
                continue
            if p.suffix.lower() != ".mp3" and p.name != "manifest.json":
                continue
            arcname = p.relative_to(REPO_DIR).as_posix()
            comp = zipfile.ZIP_STORED if p.suffix.lower() == ".mp3" else zipfile.ZIP_DEFLATED
            zf.write(p, arcname=arcname, compress_type=comp)
            archived_files += 1

        zf.write(js_manifest_path, arcname="js/voice-manifest.js", compress_type=zipfile.ZIP_DEFLATED)
        zf.write(QA_REPORT_PATH, arcname="tools/voice/qa_report.json", compress_type=zipfile.ZIP_DEFLATED)
        archived_files += 2

    zip_mb = LOCAL_ZIP_PATH.stat().st_size / (1024 * 1024)
    print(f"✅ 打包完成: {LOCAL_ZIP_PATH} ({zip_mb:.2f} MB, 共收录 {archived_files} 个文件)", flush=True)

    # 覆盖备份到 Google Drive（独立捕获 OSError，绝不因云盘配额或 FUSE 抖动阻断浏览器下载）
    if drive_mounted and DRIVE_ZIP_ROOT_COPY is not None:
        try:
            shutil.copy2(LOCAL_ZIP_PATH, DRIVE_ZIP_ROOT_COPY)
            shutil.copy2(LOCAL_ZIP_PATH, WORK_DIR / "ojousama_jp_voice_pack.zip")
            shutil.copy2(QA_REPORT_PATH, WORK_DIR / "qa_report.json")
            print(f"✅ 已覆盖备份最新完整语音包至 Google Drive: {DRIVE_ZIP_ROOT_COPY}", flush=True)
        except OSError as drive_exc:
            print(f"⚠️ Google Drive 备份写入失败 ({drive_exc})，本地压缩包完好：{LOCAL_ZIP_PATH}", flush=True)

    print(
        "\n🎉 精准重抽与全量合包圆满完成！\n"
        f"  - 硬件档位: {TIER_DESC}\n"
        f"  - 运行模式: {'增量精准重抽 (保留原装好句)' if incremental_retake_mode else '全量合成'}\n"
        f"  - 本轮合成: {qa_stats.get('succeeded', 0)} / {qa_stats.get('totalSelected', 0)} 条 "
        f"(失败: {qa_stats.get('failed', 0)} 条)\n"
        f"  - 成品合包: {synced_mp3_count} / {len(valid_rel_paths)} 个 MP3 -> {LOCAL_ZIP_PATH} ({zip_mb:.2f} MB)\n"
        "  - 使用方法: 解压覆盖到游戏仓库根目录 (`unzip -o ojousama_jp_voice_pack.zip -d .`)",
        flush=True,
    )

    try:
        from google.colab import files  # type: ignore

        print("⬇️ 正在触发浏览器自动下载 ojousama_jp_voice_pack.zip ...", flush=True)
        files.download(str(LOCAL_ZIP_PATH))
    except Exception as exc:
        fallback_loc = DRIVE_ZIP_ROOT_COPY if (drive_mounted and DRIVE_ZIP_ROOT_COPY) else LOCAL_ZIP_PATH
        print(f"ℹ️ 浏览器自动下载未触发 ({exc})，请直接从 {fallback_loc} 获取。", flush=True)

finally:
    if serve_proc is not None:
        stop_server_process(serve_proc)
    if serve_log_fp is not None:
        try:
            serve_log_fp.close()
        except Exception:
            pass